# ⚡ SURESH.GPT Remote GPU Server (Google Colab T4)

This notebook turns Google Colab into a high-speed GPU inference server for your local **SURESH.GPT Platform**.

### How It Works:
1. **Colab GPU**: Runs `Qwen/Qwen2.5-1.5B-Instruct` on the free NVIDIA T4 GPU (15GB VRAM).
2. **Your Local Machine**: Runs the Streamlit UI, Document RAG, FAISS index, and SQLite database with 0% CPU lag.
3. Tokens stream directly from Colab to your local browser in real time at **30–50 tokens/sec**.

---
### Step 1: Verify Free T4 GPU
Ensure you have GPU enabled: **Runtime** > **Change runtime type** > **T4 GPU** > **Save**.

In [ ]:
!nvidia-smi

### Step 2: Install Lightweight Inference Dependencies
No zip file upload required! Everything installs in ~30 seconds.

In [ ]:
!pip install -q transformers torch accelerate fastapi uvicorn nest_asyncio
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Dependencies installed successfully!")

### Step 3: Load Qwen2.5 on GPU & Start Streaming Server
Run this cell. It will load `Qwen/Qwen2.5-1.5B-Instruct` into VRAM and print your **Public GPU Server URL**.

Copy that URL and paste it into your local SURESH.GPT app under **🎛 Chat Parameters** > **Colab GPU Server URL**!

In [ ]:
import os
import time
import threading
import torch
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import List, Dict, Any, Optional
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

nest_asyncio.apply()

# 1. Model Configuration
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"⏳ Loading {MODEL_NAME} onto {DEVICE} ({DTYPE})...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    trust_remote_code=True
).to(DEVICE)
model.eval()
print(f"✅ Model loaded successfully on {DEVICE}!")

# 2. Define FastAPI Server
app = FastAPI(title="SURESH.GPT Colab GPU Server")

class ChatRequest(BaseModel):
    messages: List[Dict[str, str]]
    temperature: float = 0.7
    max_tokens: int = 512
    top_p: float = 0.9
    top_k: int = 50

@app.get("/health")
def health():
    return {
        "status": "ok",
        "device": DEVICE,
        "model": MODEL_NAME,
        "cuda_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    }

@app.post("/stream")
def stream_chat(req: ChatRequest):
    formatted_prompt = tokenizer.apply_chat_template(
        req.messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(DEVICE)
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    gen_kwargs = dict(
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask"),
        streamer=streamer,
        max_new_tokens=req.max_tokens,
        temperature=req.temperature if req.temperature > 0 else 1.0,
        top_p=req.top_p if req.temperature > 0 else 1.0,
        top_k=req.top_k if req.temperature > 0 else 1,
        do_sample=(req.temperature > 0),
        pad_token_id=tokenizer.eos_token_id
    )

    thread = threading.Thread(target=model.generate, kwargs=gen_kwargs)
    thread.daemon = True
    thread.start()

    def token_generator():
        for token in streamer:
            yield token

    return StreamingResponse(token_generator(), media_type="text/plain")

# 3. Run Server in Background Thread
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("🚀 FastAPI server is running on http://127.0.0.1:8000!")

# 4. Launch Cloudflare Tunnel and Print Link
print("\n" + "="*65)
print("👉 COPY THE PUBLIC HTTPS URL BELOW AND PASTE INTO YOUR LOCAL APP:")
print("="*65 + "\n")
!cloudflared tunnel --url http://127.0.0.1:8000